# Exercise 3 — Simple Dispatch Model


## Task

Build an optimization model for power generation in a market. Several firms offer generation capacity at a specified marginal cost in each time step. Demand is known for every time step.

The model must:

- minimize total generation cost;
- meet demand **exactly** in every time step; and
- allow any quantity between zero and the offered capacity to be dispatched.

First, formulate the optimization problem algebraically.

Then implement

```julia
gen_simp_dis(cap, costs, demand, optimizer)
```

with the following inputs:

- `cap::Dict{String, Vector{Float64}}`: offered capacity of each firm by time step;
- `costs::Dict{String, Vector{Float64}}`: generation cost of each firm by time step;
- `demand::Vector{Float64}`: demand by time step;
- `optimizer`: the optimizer constructor.

The function should return the **unsolved** JuMP model. Test your implementation with the supplied test data; the optimal objective value should be `16`.


## Supplied test data


In [1]:
using JuMP, HiGHS

costs = Dict(
    "A" => [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "B" => [2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
    "C" => [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "D" => [2.0, 2.0, 2.0, 2.0, 2.0, 2.0],
)

cap = Dict(
    "A" => [1.0, 2.0, 1.0, 0.0, 0.0, 0.0],
    "B" => [2.0, 1.0, 2.0, 0.0, 0.0, 0.0],
    "C" => [0.0, 0.0, 0.0, 1.0, 2.0, 1.0],
    "D" => [0.0, 0.0, 0.0, 2.0, 1.0, 2.0],
)

demand = [1.0, 2.0, 3.0, 1.0, 2.0, 3.0]

6-element Vector{Float64}:
 1.0
 2.0
 3.0
 1.0
 2.0
 3.0

## Reference implementation


In [2]:
function gen_simp_dis(cap, costs, demand, optimizer)

    firms = keys(cap)
    hours = 1:length(demand)
    m = Model(optimizer)
    @variable(m, x[firms, hours] >= 0)

    @objective(m, Min, sum(costs[f][h] * x[f, h] for f in firms, h in hours))

    @constraint(m, demand_match[h in hours], sum(x[f, h] for f in firms) == demand[h])
    @constraint(m, cap_lim[f in firms, h in hours], x[f, h] <= cap[f][h])

    return m
end


gen_simp_dis (generic function with 1 method)

## Test


In [3]:
m = gen_simp_dis(cap, costs, demand, HiGHS.Optimizer)
optimize!(m)
@assert objective_value(m) == 16


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 30 rows; 24 cols; 48 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 2e+00]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 3e+00]
Presolving model
4 rows, 5 cols, 5 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-30); columns 0(-24); nonzeros 0(-48) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     1.4000032127e+01 Pr: 1(2) 0.0s
          2     1.6000000000e+01 Pr: 0(0) 0.0s
Required 2 simplex iterations after postsolve

Model status        : Optimal
Simplex   iterations: 2
Objective value     :  1.6000000000e+01
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
